# 🧹 Task 4: Automated Data Cleaning & Reporting Workflows

## 📌 Objective
Automate the ingestion, sanitization, and reporting of messy real-world operational datasets.

### 🔑 Key Data Preprocessing Objectives:
1. **Deduplication:** Remove redundant order IDs and identical rows.
2. **Text Normalization:** Trim whitespace, correct erratic casing, and map non-standard category codes.
3. **Missing Value Imputation:** Impute missing names, regions, and calculate missing sales from unit price, quantity, and discount.
4. **Type Casting & Date Formatting:** Sanitize currency symbols (`$`, `USD`, commas) and convert multi-format dates to ISO `YYYY-MM-DD`.
5. **Automated Reporting:** Generate automated aggregated visual summaries and audit statistics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Ingest Raw Dirty Data
df_raw = pd.read_csv('raw_uncleaned_sales_data.csv')
print(f"Raw Ingested Records: {df_raw.shape[0]} | Columns: {df_raw.shape[1]}")
df_raw.head(8)

## 2. Profiling Initial Data Quality Flaws

In [2]:
print("--- Missing Values per Column ---")
print(df_raw.isnull().sum())

duplicate_count = df_raw.duplicated(subset=['Order_ID']).sum()
print(f"\nDuplicate Order_IDs detected: {duplicate_count}")

print("\nUnique Dirty Categories:")
print(df_raw['Category'].unique())

## 3. Automated Cleaning Pipeline Execution

In [3]:
# 1. Deduplication
df_clean = df_raw.drop_duplicates(subset=['Order_ID'], keep='first').copy()

# 2. Text Trimming & Normalization
df_clean['Customer_Name'] = df_clean['Customer_Name'].fillna('Valued Customer').astype(str).str.strip().str.title()

# 3. Category Standardization
def map_cat(val):
    s = str(val).lower().strip()
    if 'tech' in s: return 'Technology'
    if 'furn' in s: return 'Furniture'
    return 'Office Supplies'
df_clean['Category'] = df_clean['Category'].apply(map_cat)

# 4. Sanitize Numeric Fields
df_clean['Unit_Price'] = df_clean['Unit_Price'].astype(str).str.replace(r'[\$,USD ]', '', regex=True).astype(float)
df_clean['Discount'] = df_clean['Discount'].astype(str).str.replace('%', '').str.strip()
df_clean['Discount'] = pd.to_numeric(df_clean['Discount'], errors='coerce').fillna(0.0)
df_clean['Discount'] = df_clean['Discount'].apply(lambda d: d/100.0 if d > 1.0 else d)

# Quantity fix (non-positive to 1)
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce').fillna(1)
df_clean['Quantity'] = df_clean['Quantity'].apply(lambda q: 1 if q <= 0 else int(q))

# Sales Amount Imputation
computed_sales = (df_clean['Unit_Price'] * df_clean['Quantity'] * (1 - df_clean['Discount'])).round(2)
df_clean['Sales_Amount'] = computed_sales

# 5. Date Standardizing
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')

print(f"Cleaned Records Verified: {len(df_clean)}")
df_clean.head()

## 4. Automated Reporting & Aggregations

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_revenue = df_clean.groupby('Category')['Sales_Amount'].sum()
axes[0].bar(cat_revenue.index, cat_revenue.values, color=['#3b82f6', '#10b981', '#f59e0b'])
axes[0].set_title('Cleaned Revenue by Category', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Revenue ($)')

reg_counts = df_clean['Region'].value_index if hasattr(df_clean['Region'], 'value_index') else df_clean['Region'].value_counts()
axes[1].pie(reg_counts.values, labels=reg_counts.index, autopct='%1.1f%%', colors=['#60a5fa', '#34d399', '#fbbf24', '#f87171'])
axes[1].set_title('Order Distribution by Region', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()